In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
import joblib
import sys, sklearn, xgboost

print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__)
print("Pandas      :", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("XGBoost     :", xgboost.__version__)
print("Joblib      :", joblib.__version__)



## Generating dataset

In [ ]:
np.random.seed(42)

c = 3e8  # speed of light (m/s)
N_SAMPLES = 10000

rows = []

for _ in range(N_SAMPLES):
    freq = np.random.uniform(1, 10)                 # GHz
    er = np.random.uniform(2.2, 10.2)                # dielectric constant
    h_mm = np.random.uniform(0.8, 3.2)               # substrate height (mm)
    loss_tangent = np.random.uniform(0.0009, 0.03)
    copper = np.random.choice([0.017, 0.035, 0.070]) # copper thickness (mm)

    h = h_mm / 1000
    f = freq * 1e9

    # Patch width
    W = (c / (2 * f)) * np.sqrt(2 / (er + 1))

    # Effective dielectric constant
    eeff = ((er + 1) / 2) + ((er - 1) / 2) * (1 / np.sqrt(1 + 12 * h / W))

    # Effective length and length extension
    Leff = c / (2 * f * np.sqrt(eeff))
    deltaL = (
        0.412 * h * ((eeff + 0.3) * ((W / h) + 0.264))
        / ((eeff - 0.258) * ((W / h) + 0.8))
    )

    # Patch length
    L = Leff - 2 * deltaL

    # Ground plane dimensions
    Wg = W + 6 * h
    Lg = L + 6 * h

    rows.append([
        freq, er, h_mm, loss_tangent, copper,
        W * 1000, L * 1000, Wg * 1000, Lg * 1000
    ])

df = pd.DataFrame(rows, columns=[
    "Frequency_GHz",
    "Dielectric_Constant",
    "Height_mm",
    "Loss_Tangent",
    "Copper_Thickness_mm",
    "Patch_Width_mm",
    "Patch_Length_mm",
    "Ground_Width_mm",
    "Ground_Length_mm",
])

df.to_csv("antenna_dataset.csv", index=False)
print(f"Generated {len(df)} rows.")
df.head()

##  Training model

In [ ]:
FEATURES = [
    "Frequency_GHz",
    "Dielectric_Constant",
    "Height_mm",
    "Loss_Tangent",
    "Copper_Thickness_mm",
]

TARGETS = [
    "Patch_Width_mm",
    "Patch_Length_mm",
    "Ground_Width_mm",
    "Ground_Length_mm",
]

X = df[FEATURES]
y = df[TARGETS]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

xgb = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

model = MultiOutputRegressor(xgb)
model.fit(X_train, y_train)

print("Training complete.")

In [ ]:
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R²   : {r2:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")

In [ ]:
joblib.dump(model, "antenna_model.pkl")
print("Saved antenna_model.pkl")

In [ ]:
reloaded = joblib.load("antenna_model.pkl")

sample = pd.DataFrame([{
    "Frequency_GHz": 2.4,
    "Dielectric_Constant": 4.4,
    "Height_mm": 1.6,
    "Loss_Tangent": 0.02,
    "Copper_Thickness_mm": 0.035,
}])

pred = reloaded.predict(sample)[0]
print("Patch Width  :", round(pred[0], 3), "mm")
print("Patch Length :", round(pred[1], 3), "mm")
print("Ground Width :", round(pred[2], 3), "mm")
print("Ground Length:", round(pred[3], 3), "mm")